## Step 1: Install and imports

In [ ]:
!pip install -q torch scikit-learn pandas numpy matplotlib seaborn

## Step 2: Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from google.colab import files

## Step 3: Upload fused embeddings CSV

In [ ]:
uploaded = files.upload()
df = pd.read_csv("reef_fused_embeddings.csv")
print("Shape:", df.shape)
print("Label distribution:")
print(df["label"].value_counts().sort_index())

## Step 4: Dataset

In [ ]:
# ── Remap 5 classes → 3 classes ─────────────────────────────
# BAA-0        -> 0 (No stress)
# BAA-1, BAA-2 -> 1 (Moderate)
# BAA-3, BAA-4 -> 2 (Severe)
LABEL_MAP = {0: 0, 1: 1, 2: 1, 3: 2, 4: 2}
df["label"] = df["label"].map(LABEL_MAP)
print("Remapped label distribution:")
print(df["label"].value_counts().sort_index())


# ── Normalise CNN and NLP embeddings separately ──────────────
from sklearn.preprocessing import StandardScaler

cnn_cols = [f"emb_{i}" for i in range(256)]
nlp_cols = [f"nlp_emb_{i+1}" for i in range(384)]

scaler_cnn = StandardScaler()
scaler_nlp = StandardScaler()

train_idx = df[df["split"] == "train"].index
test_idx  = df[df["split"] == "test"].index

df.loc[train_idx, cnn_cols] = scaler_cnn.fit_transform(df.loc[train_idx, cnn_cols])
df.loc[test_idx,  cnn_cols] = scaler_cnn.transform(df.loc[test_idx, cnn_cols])

df.loc[train_idx, nlp_cols] = scaler_nlp.fit_transform(df.loc[train_idx, nlp_cols])
df.loc[test_idx,  nlp_cols] = scaler_nlp.transform(df.loc[test_idx, nlp_cols])

print("CNN and NLP embeddings normalised.")

CNN_DIM = 256
NLP_DIM = 384

cnn_cols = [f"emb_{i}" for i in range(CNN_DIM)]
nlp_cols = [f"nlp_emb_{i+1}" for i in range(NLP_DIM)]

class FusionDataset(Dataset):
    def __init__(self, df):
        self.cnn = torch.tensor(df[cnn_cols].values, dtype=torch.float32)
        self.nlp = torch.tensor(df[nlp_cols].values, dtype=torch.float32)
        self.labels = torch.tensor(df["label"].values, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.cnn[idx], self.nlp[idx], self.labels[idx]

train_df = df[df["split"] == "train"].reset_index(drop=True)
test_df  = df[df["split"] == "test"].reset_index(drop=True)

train_ds = FusionDataset(train_df)
test_ds  = FusionDataset(test_df)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=64, shuffle=False)

print(f"Train: {len(train_ds)}  |  Test: {len(test_ds)}")

## Step 5: Gated fusion model

Instead of 2 scalar attention weights, a **256-dim sigmoid gate** independently controls how much CNN vs NLP contributes to each dimension of the fused vector.

In [ ]:
class GatedFusionModel(nn.Module):
    def __init__(self, cnn_dim=256, nlp_dim=384, hidden_dim=256, num_classes=3, dropout=0.3):
        super().__init__()

        # Project each modality into shared hidden space
        self.cnn_proj = nn.Sequential(
            nn.Linear(cnn_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.nlp_proj = nn.Sequential(
            nn.Linear(nlp_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Gate: per-dimension sigmoid weight (hidden_dim values, not just 2 scalars)
        # Gate is conditioned on BOTH modalities
        self.gate = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Sigmoid()   # output in (0, 1) per dimension
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_classes)
        )

    def forward(self, cnn_emb, nlp_emb):
        cnn_h = self.cnn_proj(cnn_emb)   # (B, hidden_dim)
        nlp_h = self.nlp_proj(nlp_emb)   # (B, hidden_dim)

        # Gate: per-dimension blend of CNN and NLP
        gate  = self.gate(torch.cat([cnn_h, nlp_h], dim=-1))  # (B, hidden_dim)
        fused = gate * cnn_h + (1 - gate) * nlp_h             # (B, hidden_dim)

        return self.classifier(fused), gate

## Step 6: Training

Fixes applied:
- **Weighted loss** — down-weights overrepresented BAA-1 and BAA-4
- **Early stopping** — stops if val accuracy doesn't improve for 15 epochs
- **Best checkpoint** — saves the model at its highest val accuracy

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ── Class weights (inverse frequency) ────────────────────────
label_counts = torch.tensor(
    [train_df["label"].value_counts().sort_index().values],
    dtype=torch.float32
).squeeze()
class_weights = (1.0 / label_counts)
class_weights = class_weights / class_weights.sum() * len(label_counts)
class_weights = class_weights.to(device)
print("Class weights:", class_weights.cpu().numpy().round(3))

model     = GatedFusionModel(num_classes=3).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# ── Early stopping config ─────────────────────────────────────
EPOCHS        = 100
PATIENCE      = 15
best_val_acc  = 0.0
patience_ctr  = 0
best_state    = None
train_losses, val_accs = [], []

for epoch in range(EPOCHS):
    # Train
    model.train()
    total_loss = 0
    for cnn_b, nlp_b, labels in train_loader:
        cnn_b, nlp_b, labels = cnn_b.to(device), nlp_b.to(device), labels.to(device)
        optimizer.zero_grad()
        logits, _ = model(cnn_b, nlp_b)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()

    # Validate
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for cnn_b, nlp_b, labels in test_loader:
            cnn_b, nlp_b, labels = cnn_b.to(device), nlp_b.to(device), labels.to(device)
            logits, _ = model(cnn_b, nlp_b)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

    acc = correct / total
    train_losses.append(total_loss / len(train_loader))
    val_accs.append(acc)

    # Save best
    if acc > best_val_acc:
        best_val_acc = acc
        patience_ctr = 0
        best_state   = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_ctr += 1

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {train_losses[-1]:.4f} | Val Acc: {acc:.4f} | Best: {best_val_acc:.4f} | Patience: {patience_ctr}/{PATIENCE}")

    if patience_ctr >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}. Best val acc: {best_val_acc:.4f}")
        break

# Restore best weights
model.load_state_dict(best_state)
print(f"\nRestored best model (val acc: {best_val_acc:.4f})")

## Step 7: Evaluate

In [ ]:
model.eval()
all_preds, all_labels, all_gates = [], [], []

with torch.no_grad():
    for cnn_b, nlp_b, labels in test_loader:
        cnn_b, nlp_b = cnn_b.to(device), nlp_b.to(device)
        logits, gate = model(cnn_b, nlp_b)
        all_preds.append(logits.argmax(dim=1).cpu())
        all_labels.append(labels)
        all_gates.append(gate.cpu())

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
all_gates  = torch.cat(all_gates).numpy()

print("\n=== Gated Fusion Classification Report ===")
print(classification_report(all_labels, all_preds,
      target_names=["No Stress", "Moderate", "Severe"]))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Stress","Moderate","Severe"],
            yticklabels=["No Stress","Moderate","Severe"])
plt.title("Confusion Matrix — Gated Fusion Model")
plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("/content/confusion_matrix_gated.png", dpi=150)
plt.show()

# Gate analysis — mean gate value per dimension tells us CNN vs NLP blend
mean_gate = all_gates.mean(axis=0)   # (hidden_dim,)
print(f"\nMean gate value (per dim average): {mean_gate.mean():.3f}")
print(f"  gate → 1.0 means CNN dominates that dimension")
print(f"  gate → 0.0 means NLP dominates that dimension")
print(f"  Dims where CNN dominates (gate > 0.7): {(mean_gate > 0.7).sum()}")
print(f"  Dims where NLP dominates (gate < 0.3): {(mean_gate < 0.3).sum()}")
print(f"  Mixed dims (0.3-0.7)                 : {((mean_gate >= 0.3) & (mean_gate <= 0.7)).sum()}")

## Step 8: Training curves

In [ ]:
best_epoch = int(np.argmax(val_accs))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses)
ax1.set_title("Training Loss"); ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")

ax2.plot(val_accs, label="Val Accuracy")
ax2.axvline(best_epoch, color="red", linestyle="--", label=f"Best epoch {best_epoch+1} ({best_val_acc:.3f})")
ax2.set_title("Validation Accuracy"); ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy")
ax2.legend()

plt.tight_layout()
plt.savefig("/content/training_curves.png", dpi=150)
plt.show()
print(f"Best epoch: {best_epoch+1}  |  Best val acc: {best_val_acc:.4f}")

## Step 9: Save model

In [ ]:
torch.save(model.state_dict(), "/content/attention_fusion_model.pth")
print("Model saved.")

from google.colab import files as colab_files
colab_files.download("/content/attention_fusion_model.pth")
colab_files.download("/content/confusion_matrix.png")
colab_files.download("/content/training_curves.png")

## Step 10: XGBoost baseline

Train XGBoost on the same 640-dim fused embeddings and compare directly against the attention fusion model.

In [ ]:
!pip install -q xgboost
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# ── Prepare data ───────────────────────────────────────────
cnn_cols = [f"emb_{i}" for i in range(256)]
nlp_cols = [f"nlp_emb_{i+1}" for i in range(384)]
all_feat_cols = cnn_cols + nlp_cols

X_train = train_df[all_feat_cols].values
y_train = train_df["label"].values
X_test  = test_df[all_feat_cols].values
y_test  = test_df["label"].values

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

# ── Train XGBoost ──────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=100
)

# ── Evaluate ───────────────────────────────────────────────
xgb_preds = xgb.predict(X_test)
xgb_acc   = accuracy_score(y_test, xgb_preds)

print("\n=== XGBoost Classification Report ===")
print(classification_report(y_test, xgb_preds,
      target_names=["No Stress", "Moderate", "Severe"]))

# Confusion matrix
cm = confusion_matrix(y_test, xgb_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens",
            xticklabels=["No Stress","Moderate","Severe"],
            yticklabels=["No Stress","Moderate","Severe"])
plt.title("Confusion Matrix — XGBoost")
plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("/content/confusion_matrix_xgb.png", dpi=150)
plt.show()

## Step 11: Compare Attention Fusion vs XGBoost

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

# Gated fusion metrics
gated_acc      = accuracy_score(all_labels, all_preds)
gated_macro    = f1_score(all_labels, all_preds, average="macro")
gated_weighted = f1_score(all_labels, all_preds, average="weighted")

# XGBoost metrics
xgb_acc      = accuracy_score(y_test, xgb_preds)
xgb_macro    = f1_score(y_test, xgb_preds, average="macro")
xgb_weighted = f1_score(y_test, xgb_preds, average="weighted")

print("=" * 52)
print(f"{'Metric':<20} {'Gated Fusion':>12} {'XGBoost':>12}")
print("=" * 52)
print(f"{'Accuracy':<20} {gated_acc:>12.4f} {xgb_acc:>12.4f}")
print(f"{'Macro F1':<20} {gated_macro:>12.4f} {xgb_macro:>12.4f}")
print(f"{'Weighted F1':<20} {gated_weighted:>12.4f} {xgb_weighted:>12.4f}")
print("=" * 52)

winner = "Gated Fusion" if gated_acc >= xgb_acc else "XGBoost"
print(f"\nWinner: {winner}")

# Bar chart
metrics     = ["Accuracy", "Macro F1", "Weighted F1"]
gated_vals  = [gated_acc, gated_macro, gated_weighted]
xgb_vals    = [xgb_acc,   xgb_macro,   xgb_weighted]

x = np.arange(len(metrics))
w = 0.35
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w/2, gated_vals, w, label="Gated Fusion", color="steelblue")
ax.bar(x + w/2, xgb_vals,   w, label="XGBoost",      color="seagreen")
ax.set_xticks(x); ax.set_xticklabels(metrics)
ax.set_ylim(0.7, 1.0)
ax.set_ylabel("Score"); ax.set_title("Gated Fusion vs XGBoost")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("/content/model_comparison_gated.png", dpi=150)
plt.show()

from google.colab import files as colab_files
colab_files.download("/content/confusion_matrix_gated.png")
colab_files.download("/content/model_comparison_gated.png")